In [24]:
import warnings
warnings.filterwarnings('ignore')

In [25]:
# STEFANOS: Disable pip install
# !pip install lazypredict
# !pip install --upgrade pandas
# !pip install fast-tabnet
# !pip install fastai
# !pip install pandas-profiling

In [26]:
#A program that takes a csv and trains models on it. Streamlined model selection.
#==============================================================================

# STEFANOS: Disable unneeded modules
# #LazyPredict
# import lazypredict
# from lazypredict.Supervised import LazyRegressor
# from lazypredict.Supervised import LazyClassifier
# #Baysian Optimization
# from bayes_opt import BayesianOptimization
#Pandas stack
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import pandas_profiling
import numpy as np
# STEFANOS: Disable unneeded modules
# #FastAI
from fastai.tabular.all import *
from fastai.tabular.core import *
# #Plots
# import matplotlib.pyplot as plt
# import seaborn as sns
#System
import os
import sys
import traceback
#Fit an xgboost model
# STEFANOS: Disable unneeded modules
# from xgboost import XGBRegressor
# from xgboost import XGBClassifier
# from xgboost import plot_importance
# from sklearn.metrics import mean_squared_error
# from sklearn.metrics import roc_auc_score
#Random
import random

#TabNet
from fast_tabnet.core import *

import shutil

In [27]:
# STEFANOS: Disable matplotlib inline
# %matplotlib inline

In [28]:
import dias.rewriter

In [29]:
# For Styling
# STEFANOS: Disbale plotting
# plt.style.use('seaborn-bright')

In [30]:
#Project Variables
#===================================================================================================
PROJECT_NAME = 'superstore'
VARIABLE_FILES = False
#Maximum amount of rows to take

## -- STEFANOS -- Remove sample

# SAMPLE_COUNT = 4000
FASTAI_LEARNING_RATE = 1e-1
AUTO_ADJUST_LEARNING_RATE = False
#Set to True automatically infer if variables are categorical or continuous
ENABLE_BREAKPOINT = True
#When trying to declare a column a continuous variable, if it fails, convert it to a categorical variable
CONVERT_TO_CAT = False
REGRESSOR = True
SEP_DOLLAR = False
SEP_COMMA = True
SHUFFLE_DATA = True

In [31]:
import time
start_time = time.time()

In [32]:
# DIAS_VERBOSE
### cell 0 ###
### GPU
input_dir = f'../input/{PROJECT_NAME}'
# STEFANOS: Change the working path
# param_dir = f'/kaggle/working/{PROJECT_NAME}'
param_dir = f'./working/{PROJECT_NAME}'
TARGET = ''
PARAM_DIR = param_dir
print(f'param_dir: {param_dir}')
if not os.path.exists(param_dir):
    os.makedirs(param_dir)
#rename any file in param_dir/file that ends with csv to data.csv
for file in os.listdir(input_dir):
    if file.endswith('.csv'):
        print('CSV!')
        if 'classification_results' not in file and 'regression_results' not in file:
            #os.rename(f'{input_dir}/{file}', f'{param_dir}/data.csv')
            shutil.copy(f'{input_dir}/{file}', f'{param_dir}/data.csv')
        #os.rename(f'{param_dir}/{file}', f'{param_dir}/data.csv')
try:
    # STEFANOS: Remove sample
#     df = pd.read_csv(f'{param_dir}/data.csv', nrows=SAMPLE_COUNT)
    df = pd.read_csv(f'{param_dir}/data.csv')
except Exception as e:
    print(f'Please place a file named data.csv in {param_dir}')

param_dir: ./working/superstore
CSV!


### Dias did not rewrite code

In [33]:
# DIAS_VERBOSE
### cell 1 ###
df

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,Second Class,Consumer,United States,Miami,Florida,33180,South,Furniture,Furnishings,25.2480,3,0.20,4.1028
9990,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Furniture,Furnishings,91.9600,2,0.00,15.6332
9991,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Technology,Phones,258.5760,2,0.20,19.3932
9992,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Office Supplies,Paper,29.6000,4,0.00,13.3200


### Dias did not rewrite code

# -- STEFANOS -- Replicate Data

In [34]:
# DIAS_VERBOSE
### cell 2 ###
factor = 10
df = pd.concat([df]*factor)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99940 entries, 0 to 9993
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Ship Mode     99940 non-null  object 
 1   Segment       99940 non-null  object 
 2   Country       99940 non-null  object 
 3   City          99940 non-null  object 
 4   State         99940 non-null  object 
 5   Postal Code   99940 non-null  int64  
 6   Region        99940 non-null  object 
 7   Category      99940 non-null  object 
 8   Sub-Category  99940 non-null  object 
 9   Sales         99940 non-null  float64
 10  Quantity      99940 non-null  int64  
 11  Discount      99940 non-null  float64
 12  Profit        99940 non-null  float64
dtypes: float64(3), int64(2), object(8)
memory usage: 10.7+ MB


### Dias did not rewrite code

In [42]:
# DIAS_VERBOSE
### cell 3 ###
if SEP_DOLLAR:
    #For every column in df_copy, if the column contains a $, make a new column with the value without the $
    for col in df.columns:
        if '$' in df[col].to_string():
            df[col + '_no_dollar'] = df[col].str.replace('$', '').str.replace(',', '')
            #Try to convert this new column to a numeric type
            try:
                df[col + '_no_dollar'] = df[col + '_no_dollar'].apply(pd.to_numeric, errors='coerce').dropna()
            except Exception:
                print(f'{col} can not be converted to a float!')


if SEP_COMMA:
    #For every column in df_copy, if the column contains a %, make a new column with the value without the %
    for col in df.columns:
        if '%' in df[col].to_string() or ',' in df[col].to_string():
            df[col + '_processed'] = df[col].apply(str).str.replace('%', '').str.replace(',', '')
            #Try to convert this new column to a numeric type
            try:
                df[col + '_processed'] = df[col + '_processed'].apply(pd.to_numeric, errors='coerce').dropna()
            except Exception:
                print(f'{col} can not be converted to a float!')


### Dias rewrote code:
<br />

```python
def _REWR_index_contains(index, s):
    if index.dtype == np.int64:
        try:
            i = int(s)
            return len(index.loc[i]) > 0
        except:
            return False
    else:
        return index.astype(str).str.contains(s).any()
if SEP_DOLLAR:
    for col in df.columns:
        if (df[col].astype(str).str.contains('$').any() or
            _REWR_index_contains(df[col].index, '$') if type(df) == pd.
            DataFrame else '$' in df[col].to_string()):
            df[col + '_no_dollar'] = df[col].str.replace('$', '').str.replace(
                ',', '')
            try:
                df[col + '_no_dollar'] = df[col + '_no_dollar'].apply(pd.
                    to_numeric, errors='coerce').dropna()
            except Exception:
                print(f'{col} can not be converted to a float!')
def _REWR_index_contains(index, s):
    if index.dtype == np.int64:
        try:
            i = int(s)
            return len(index.loc[i]) > 0
        except:
            return False
    else:
        return index.astype(str).str.contains(s).any()
if SEP_COMMA:
    for col in df.columns:
        if (df[col].astype(str).str.contains('%').any() or
            _REWR_index_contains(df[col].index, '%') if type(df) == pd.
            DataFrame else '%' in df[col].to_string()) or (df[col].astype(
            str).str.contains(',').any() or _REWR_index_contains(df[col].
            index, ',') if type(df) == pd.DataFrame else ',' in df[col].
            to_string()):
            df[col + '_processed'] = df[col].apply(str).str.replace('%', ''
                ).str.replace(',', '')
            try:
                df[col + '_processed'] = df[col + '_processed'].apply(pd.
                    to_numeric, errors='coerce').dropna()
            except Exception:
                print(f'{col} can not be converted to a float!')

```


In [13]:
### cell 4 ###
df

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,Second Class,Consumer,United States,Miami,Florida,33180,South,Furniture,Furnishings,25.2480,3,0.20,4.1028
9990,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Furniture,Furnishings,91.9600,2,0.00,15.6332
9991,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Technology,Phones,258.5760,2,0.20,19.3932
9992,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Office Supplies,Paper,29.6000,4,0.00,13.3200


In [36]:
# DIAS_VERBOSE
### cell 5 ###
df.isna().sum()

Ship Mode       0
Segment         0
Country         0
City            0
State           0
Postal Code     0
Region          0
Category        0
Sub-Category    0
Sales           0
Quantity        0
Discount        0
Profit          0
dtype: int64

### Dias did not rewrite code

In [37]:
# DIAS_VERBOSE
### cell 6 ###
df.profile_report()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

### Dias did not rewrite code

In [38]:
# DIAS_VERBOSE
### cell 7 ###
# STEFANOS: Disable plotting
# sns.heatmap(df.corr())
df.select_dtypes(include=['number']).corr()

,Postal Code,Sales,Quantity,Discount,Profit
Postal Code,1.000000,-0.023854,0.012761,0.058443,-0.029961
Sales,-0.023854,1.000000,0.200795,-0.028190,0.479064
Quantity,0.012761,0.200795,1.000000,0.008623,0.066253
Discount,0.058443,-0.028190,0.008623,1.000000,-0.219487
Profit,-0.029961,0.479064,0.066253,-0.219487,1.000000


### Dias did not rewrite code

In [17]:
### cell 8 ###
df.head()

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


In [39]:
# DIAS_VERBOSE
### cell 9 ###
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Postal Code,99940.0,55190.379428,32062.249571,1040.000,23223.000,56430.5000,90008.000,99301.000
Sales,99940.0,229.858001,623.217037,0.444,17.280,54.4900,209.940,22638.480
Quantity,99940.0,3.789574,2.225009,1.000,2.000,3.0000,5.000,14.000
Discount,99940.0,0.156203,0.206443,0.000,0.000,0.2000,0.200,0.800
Profit,99940.0,28.656896,234.249559,-6599.978,1.728,8.6665,29.364,8399.976


### Dias did not rewrite code

In [40]:
# DIAS_VERBOSE
### cell 10 ###
df.columns

Index(['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Postal Code',
       'Region', 'Category', 'Sub-Category', 'Sales', 'Quantity', 'Discount',
       'Profit'],
      dtype='object')

### Dias did not rewrite code

In [41]:
# DIAS_VERBOSE
target = ''
target_str = ''
#The column closest to the end isPARAM_DIR the target variable that can be represented as a float is the target variable
targets = []
#Loop through every possible target column (Continuous)
for i in range(len(df.columns)-1, 0, -1):
    try:
        df[df.columns[i]] = df[df.columns[i]].apply(pd.to_numeric, errors='coerce').dropna()
        target = df.columns[i]
        target_str = target.replace('/', '-')
    except:
        continue
    print(f'Target Variable: {target}')
    #Will be determined by the file name


    #===================================================================================================

    #Create project config files if they don't exist.
    if not os.path.exists(param_dir):
        #create param_dir
        os.makedirs(PARAM_DIR)
    if not os.path.exists(f'{PARAM_DIR}/cats.txt'):
        #create param_dir
        with open(f'{PARAM_DIR}/cats.txt', 'w') as f:
            f.write('')
    if not os.path.exists(f'{PARAM_DIR}/conts.txt'):
        #create param_dir
        with open(f'{PARAM_DIR}/conts.txt', 'w') as f:
            f.write('')
    if not os.path.exists(f'{PARAM_DIR}/cols_to_delete.txt'):
        with open(f'{PARAM_DIR}/cols_to_delete.txt', 'w') as f:
            f.write('')

    df = df.drop_duplicates()
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # workaround for fastai/pytorch bug where bool is treated as object and thus erroring out.
    for n in df:
        if pd.api.types.is_bool_dtype(df[n]):
            df[n] = df[n].astype('uint8')

    with open(f'{PARAM_DIR}/cols_to_delete.txt', 'r') as f:
        cols_to_delete = f.read().splitlines()
    for col in cols_to_delete:
        try:
            del(df[col])
        except:
            pass
    #try to fill in missing values now, otherwise FastAI will do it for us later
    try:
        df = df.fillna(0)
    except:
        pass
    #print missing values
    #print(df.isna().sum().sort_values(ascending=False))
    #shrink df as much as possible
    df = df_shrink(df)


    #print types inside of df
    #print(df.dtypes)


    #Auto detect categorical and continuous variables
    #==============================================================================
    likely_cat = {}
    for var in df.columns:
        likely_cat[var] = 1.*df[var].nunique()/df[var].count() < 0.05 #or some other threshold

    cats = [var for var in df.columns if likely_cat[var]]
    conts = [var for var in df.columns if not likely_cat[var]]

    #remove target from lists
    try:
        conts.remove(target)
        cats.remove(target)
    except:
        pass
    #Convert target to float
    df[target] = df[target].apply(pd.to_numeric, errors='coerce').dropna()

    print('CATS=====================')
    print(cats)
    print('CONTS=====================')
    print(conts)

    # TODO(sahil): Continue from below

    #Populate categorical and continuous lists
    #==============================================================================

    if VARIABLE_FILES == True:
        with open(f'{PARAM_DIR}/cats.txt', 'r') as f:
            cats = f.read().splitlines()

        with open(f'{PARAM_DIR}/conts.txt', 'r') as f:
            conts = f.read().splitlines()

    #==============================================================================

    #==============================================================================
    procs = [Categorify, FillMissing, Normalize]
    #print(df.describe().T)
    # STEFANOS: Remove samples
#     df = df[0:SAMPLE_COUNT]
    splits = RandomSplitter()(range_of(df))

    print((len(cats)) + len(conts))
    #conts = []

    #Convert cont variables to floats
    #==============================================================================

    #Convert cont variables to floats
    #==============================================================================

    for var in conts:
        try:
            df[var] = df[var].apply(pd.to_numeric, errors='coerce').dropna()
        except:
            print(f'Could not convert {var} to float.')
            pass

    #==============================================================================

    #Experimental logic to add columns one-by-one to find a breakpoint
    #==============================================================================
    if ENABLE_BREAKPOINT == True:
        temp_procs = [Categorify, FillMissing]
        print('Looping through continuous variables to find breakpoint')
        cont_list = []
        for cont in conts:
            focus_cont = cont
            cont_list.append(cont)
            #print(focus_cont)
            try:
                to = TabularPandas(df, procs=procs, cat_names=cats, cont_names=cont_list, y_names=target, y_block=RegressionBlock(), splits=splits)
                del(to)
            except:
                print('Error with ', focus_cont)
                #remove focus_cont from list
                cont_list.remove(focus_cont)
                #traceback.print_exc()
                continue
        #convert all continuous variables to floats
        for var in cont_list:
            try:
                df[var] = df[var].apply(pd.to_numeric, errors='coerce').dropna()
            except:
                print(f'Could not convert {var} to float.')
                cont_list.remove(var)
                if CONVERT_TO_CAT == True:
                    cats.append(var)
                pass
        print(f'Continuous variables that made the cut : {cont_list}')
        print(f'Categorical variables that made the cut : {cats}')
        #shrink df as much as possible
        df = df_shrink(df)
        #print(df.dtypes)

    #==============================================================================

    #Creating tabular object + quick preprocessing
    #==============================================================================
    to = None
    if REGRESSOR == True:
        try:
            to = TabularPandas(df, procs, cats, conts, target, y_block=RegressionBlock(), splits=splits)
        except:
            conts = []
            to = TabularPandas(df, procs, cats, conts, target, y_block=RegressionBlock(), splits=splits)
    else:
        try:
            to = TabularPandas(df, procs, cats, conts, target, splits=splits)
        except:
            conts = []
            to = TabularPandas(df, procs, cats, conts, target, splits=splits)

Target Variable: Profit
CATS=====================
['Ship Mode', 'Segment', 'Country', 'State', 'Region', 'Category', 'Sub-Category', 'Quantity', 'Discount']
CONTS=====================
['City', 'Postal Code', 'Sales']
12
Looping through continuous variables to find breakpoint
Continuous variables that made the cut : ['City', 'Postal Code', 'Sales']
Categorical variables that made the cut : ['Ship Mode', 'Segment', 'Country', 'State', 'Region', 'Category', 'Sub-Category', 'Quantity', 'Discount', 'City_na']
Target Variable: Discount
CATS=====================
['Ship Mode', 'Segment', 'Country', 'State', 'Region', 'Category', 'Sub-Category', 'Quantity', 'Discount']
CONTS=====================
['City', 'Postal Code', 'Sales', 'Profit']
13
Looping through continuous variables to find breakpoint
Continuous variables that made the cut : ['City', 'Postal Code', 'Sales', 'Profit']
Categorical variables that made the cut : ['Ship Mode', 'Segment', 'Country', 'State', 'Region', 'Category', 'Sub-Cate

### Dias did not rewrite code

# -- STEFANOS -- We have to disable the following too because even though it's Pandas code, it uses results from ML

In [21]:
### cell 12 ###
df.isna().sum()

Ship Mode          0
Segment         9962
Country         9962
City            9962
State           9962
Postal Code        0
Region          9962
Category        9962
Sub-Category    9962
Sales              0
Quantity           0
Discount           0
Profit             0
dtype: int64

In [22]:
end_time = time.time()

In [23]:
print("Total time in seconds", end_time - start_time)

Total time in seconds 15.05519413948059


# **To Be Continued...**